In [ ]:
import os
from diffusers import StableDiffusionPipeline
from transformers import BlipProcessor, BlipForConditionalGeneration
from torchvision import models, transforms
from PIL import Image
import torch
from collections import Counter
import pandas as pd

# --- Mappa létrehozása frame-eknek ---
os.makedirs("frames", exist_ok=True)

# --- Pipeline setup ---
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
).to("cuda")

resnet = models.resnet18(pretrained=True)
resnet.eval()

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# --- Prompt lista ---
prompts = ["cat", "dog", "forest scene", "robot", "castle"]
results = []

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

In [ ]:
# --- Kép generálás és feldolgozás ---
for i, prompt in enumerate(prompts):
    # Kép generálás
    img = pipe(prompt, guidance_scale=7.5).images[0]
    frame_path = f"frames/frame_{i:03d}.png"
    img.save(frame_path)

    # ResNet osztályozás
    input_tensor = preprocess(img).unsqueeze(0)
    with torch.no_grad():
        output = resnet(input_tensor)
    probs = torch.nn.functional.softmax(output[0], dim=0)
    top_prob, top_catid = torch.topk(probs, 3)

    # BLIP caption
    inputs = processor(img.convert("RGB"), return_tensors="pt")
    out = blip_model.generate(**inputs)
    caption = processor.decode(out[0], skip_special_tokens=True)

    # Eredmények mentése
    results.append({
        "frame": frame_path,
        "prompt": prompt,
        "caption": caption,
        "top_classes": top_catid.tolist(),
        "top_probs": top_prob.tolist()
    })

# --- Dataframe statisztika ---
df = pd.DataFrame(results)
df

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

,frame,prompt,caption,top_classes,top_probs
0,frames/frame_000.png,cat,a cat with blue eyes,"[281, 282, 285]","[0.4683203399181366, 0.2527714669704437, 0.172..."
1,frames/frame_001.png,dog,a dog with its tongue out,"[264, 263, 235]","[0.42657700181007385, 0.3314618170261383, 0.07..."
2,frames/frame_002.png,forest scene,a wooded landscape with a man and a dog by a s...,"[340, 372, 374]","[0.0612647570669651, 0.04042268916964531, 0.03..."
3,frames/frame_003.png,robot,a robot with blue eyes and a black and white b...,"[543, 778, 872]","[0.2018520087003708, 0.13277263939380646, 0.11..."
4,frames/frame_004.png,castle,a castle with a large tower and a tree,"[483, 663, 497]","[0.8904895186424255, 0.05666077136993408, 0.02..."


In [ ]:
import cv2

# Képek betöltése rendezett sorrendben
frames = sorted([f"frames/{f}" for f in os.listdir("frames") if f.endswith(".png")])
frame_example = cv2.imread(frames[0])
height, width, layers = frame_example.shape

# Videó létrehozása
video_name = "generated_video.mp4"
video = cv2.VideoWriter(video_name, cv2.VideoWriter_fourcc(*'mp4v'), 2, (width, height))  # 2 FPS

for f in frames:
    img = cv2.imread(f)
    video.write(img)

video.release()
print(f"Videó elkészült: {video_name}")

Videó elkészült: generated_video.mp4
